# Image classification 

For classification purpose, the output layer needs to be a FC layer with its output the class number. 

ViT needs to append a FC layer (head) because its strucute was originally designed for NLP; for other CNN based model, can modify the last layer in-place.

Fine-tuning can:
1. Update the parameters of the whole model
2. Only update the last layer (or certain layers). Can make these parts require_grad=False.

Notes:
1. By default, pretrained PyTorch models (huggingface) are built considering input data (N,C,H,W) as the first layer or embedding layer. Model itself is built with weights only taking a single data (C,H, W) because no need to duplicated the parameters. For example, the ViT model used in this notebook handles batches in the model embedding layer.
2. PyTorch dataloader will prepare data in correct batch (N,C,H,W). Then you can write outputs=model(batch_data). It will feed each data into the "actual" model, calculate the score per data,and ouput the results in batch.
3. Then you can use criterion to calculate a single loss score per minibatch, and update the weights per minibatch (instead of the whole dataset).
4. PyTorch-Lightning handles batch inside the trainer() class.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import os

In [2]:
from torchmetrics.image.fid import FrechetInceptionDistance
from torchmetrics.image.inception import InceptionScore
from torchmetrics.image.kid import KernelInceptionDistance
from torchmetrics.image.lpip import LearnedPerceptualImagePatchSimilarity

def cal_metrics(generate,exp):
    generate = torch.stack(generate)  
    exp = torch.stack(exp)  

    

    generate0 = (generate.clone().detach())
    exp0 = (exp.clone().detach())
    
    #[0,1] to [0,255]
    generate = (generate.clone().detach()*255).type(torch.uint8)
    exp = (exp.clone().detach()*255).type(torch.uint8)


    
    fid = FrechetInceptionDistance(feature=768)
    fid.update(generate, real=False)
    fid.update(exp, real=True)
    fid = fid.compute()


    inception = InceptionScore()
    inception.update(generate)
    inception=inception.compute()


    kid = KernelInceptionDistance(feature=768,subset_size=5)
    kid.update(generate, real=False)
    kid.update(exp, real=True)
    kid = kid.compute()

    lpips = LearnedPerceptualImagePatchSimilarity(net_type='vgg')
    lp =lpips(generate0, exp0)


    return fid,inception,kid, lp


In [3]:
import torchvision
from pipeline_utils import Evaluation
import pandas as pd
import numpy as np
from torch.utils.data import Subset


transform=transforms.Compose([transforms.Resize(224),
                              transforms.CenterCrop(224),
                              transforms.ToTensor(),
                              transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
                             ])

batch_size = 16



#looad generate data
rings_generated = []
peaks_generated = []
empty_generated = []


#getting generated rings
dataset_generate_rings = torchvision.datasets.ImageFolder(root="/lovelace/xiaoya/GenAI_dataset/xc_ring_1k_0_100_2nd_corrected", transform=transform)
idx_generate_rings = [i for i in range(len(dataset_generate_rings)) if dataset_generate_rings.imgs[i][1] == dataset_generate_rings.class_to_idx['Real']]
dataset_generate_rings_subset = Subset(dataset_generate_rings, idx_generate_rings)
dataloader_generate_rings = torch.utils.data.DataLoader(dataset_generate_rings_subset, batch_size=batch_size, shuffle=False)

for images, labels in dataloader_generate_rings:
    for x in range(len(labels)):
        rings_generated.append(images[x])



#getting generated peaks
dataset_generate_peaks = torchvision.datasets.ImageFolder(root="/lovelace/xiaoya/GenAI_dataset/xc_peaks_5k_0_100_1st", transform=transform)
idx_generate_peaks = [i for i in range(len(dataset_generate_peaks)) if dataset_generate_peaks.imgs[i][1] == dataset_generate_peaks.class_to_idx['real']]
dataset_generate_peaks_subset = Subset(dataset_generate_peaks, idx_generate_peaks)
dataloader_generate_peaks = torch.utils.data.DataLoader(dataset_generate_peaks_subset, batch_size=batch_size, shuffle=False)

for images, labels in dataloader_generate_peaks:
    for x in range(len(labels)):
        peaks_generated.append(images[x])



#getting generated empty
dataset_generate_empty = torchvision.datasets.ImageFolder(root="/lovelace/xiaoya/GenAI_dataset/xc_empty_5k_0_100_1st", transform=transform)
idx_generate_empty = [i for i in range(len(dataset_generate_empty)) if dataset_generate_empty.imgs[i][1] == dataset_generate_empty.class_to_idx['Real']]
dataset_generate_empty_subset = Subset(dataset_generate_empty, idx_generate_empty)
dataloader_generate_empty = torch.utils.data.DataLoader(dataset_generate_empty_subset, batch_size=batch_size, shuffle=False)

for images, labels in dataloader_generate_empty:
    for x in range(len(labels)):
        empty_generated.append(images[x])


print(len(rings_generated),len(peaks_generated),len(empty_generated))


# load exp data
rings_exp = []
peaks_exp = []
empty_exp = []
dataset_exp = torchvision.datasets.ImageFolder(root="/lovelace/xiaoya/GenAI_dataset/traindata_3folders", transform=transform)


#getting exp rings
idx_exp_rings = [i for i in range(len(dataset_exp)) if dataset_exp.imgs[i][1] == dataset_exp.class_to_idx['rings']]
dataset_exp_rings_subset = Subset(dataset_exp, idx_exp_rings)
dataloader_exp_rings = torch.utils.data.DataLoader(dataset_exp_rings_subset, batch_size=batch_size, shuffle=False)

for images, labels in dataloader_exp_rings:
    for x in range(len(labels)):
        rings_exp.append(images[x])

#getting exp peaks
idx_exp_peaks = [i for i in range(len(dataset_exp)) if dataset_exp.imgs[i][1] == dataset_exp.class_to_idx['peaks']]
dataset_exp_peaks_subset = Subset(dataset_exp, idx_exp_peaks)
dataloader_exp_peaks = torch.utils.data.DataLoader(dataset_exp_peaks_subset, batch_size=batch_size, shuffle=False)

for images, labels in dataloader_exp_rings:
    for x in range(len(labels)):
        peaks_exp.append(images[x])

#getting exp empty
idx_exp_empty = [i for i in range(len(dataset_exp)) if dataset_exp.imgs[i][1] == dataset_exp.class_to_idx['empty']]
dataset_exp_empty_subset = Subset(dataset_exp, idx_exp_empty)
dataloader_exp_empty = torch.utils.data.DataLoader(dataset_generate_empty_subset, batch_size=batch_size, shuffle=False)

for images, labels in dataloader_exp_empty:
    for x in range(len(labels)):
        empty_exp.append(images[x])

print(len(rings_exp),len(peaks_exp),len(empty_exp))


100 100 100
100 100 100


In [4]:
rings_fid, rings_inception, rings_kid, rings_lp = cal_metrics(rings_generated,rings_exp)
peaks_fid, peaks_inception, peaks_kid, peaks_lp = cal_metrics(peaks_generated,peaks_exp)
empty_fid, empty_inception, empty_kid, empty_lp = cal_metrics(empty_generated,empty_exp)
print("rings fid; inceptioon mean and std; kid mean and std; lpips:",rings_fid, rings_inception, rings_kid, rings_lp)
print("peaks fid; inceptioon mean and std; kid mean and std; lpips:",peaks_fid, peaks_inception, peaks_kid, peaks_lp)
print("empty fid; inceptioon mean and std; kid mean and std; lpips:",empty_fid, empty_inception, empty_kid, empty_lp)

/home/xiaoya/miniconda3/envs/ensemble_classifier/lib/python3.12/site-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `InceptionScore` will save all extracted features in buffer. For large datasets this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)  # noqa: B028
/home/xiaoya/miniconda3/envs/ensemble_classifier/lib/python3.12/site-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `Kernel Inception Distance` will save all extracted features in buffer. For large datasets this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)  # noqa: B028


rings fid; inceptioon mean and std; kid mean and std; lpips: tensor(0.6237) (tensor(1.9673), tensor(0.1862)) (tensor(0.0007), tensor(0.0005)) tensor(0.4378)
peaks fid; inceptioon mean and std; kid mean and std; lpips: tensor(0.9613) (tensor(1.4866), tensor(0.0918)) (tensor(0.0019), tensor(0.0004)) tensor(0.4094)
empty fid; inceptioon mean and std; kid mean and std; lpips: tensor(-8.0062e-08) (tensor(1.5090), tensor(0.1081)) (tensor(-2.5954e-05), tensor(0.0001)) tensor(0.)
